# Logbook
> What each node was doing when: deployments, events, and checks against the logs

Node IDs, plants and camera positions change as the setup grows, so the analysis can't rely on `node_id` alone. The logbook records the setup in two CSV files in `logbook/` at the project root:

- `deployments.csv` has one row per stretch of time during which a node watches one plant with one setup. A new row starts when a node is renamed, moved to another plant, or has its camera re-aimed. An empty `end` means the deployment is still running.
- `events.csv` has one row per action at a moment, such as a visit, a watering or a card collection. An empty `node_id` means the whole greenhouse.

The `source` column says whether a row was `recorded` by the team or `inferred` from the logs. Inferred rows are guesses to confirm or correct.

In [ ]:
#| default_exp logbook

In [ ]:
#| export
import numpy as np, pandas as pd
from speaking_plants.leaf import scene_similarity

In [ ]:
from fastcore.utils import *
from fastcore.test import *
from speaking_plants.clean import *

## Reading the logbook

In [ ]:
#| export
def read_deployments(
    path, # Path of `deployments.csv`
)->pd.DataFrame: # One row per deployment, with `start` and `end` as times, `end` missing while running
    "Read the deployments table"
    d = pd.read_csv(path, dtype=str, keep_default_na=False)
    return d.assign(start=pd.to_datetime(d.start), end=pd.to_datetime(d.end.replace('', None)))

def read_events(
    path, # Path of `events.csv`
)->pd.DataFrame: # One row per event, sorted by time
    "Read the events table"
    e = pd.read_csv(path, dtype=str, keep_default_na=False)
    return e.assign(time=pd.to_datetime(e.time)).sort_values('time', ignore_index=True)

In [ ]:
#| eval: false
log = Path('../logbook')
deps, evs = read_deployments(log/'deployments.csv'), read_events(log/'events.csv')
deps

,deployment_id,node_id,start,end,plant_id,site,camera,air_sensor,source,notes
0,d01,SP_01,2026-09-23 11:58:00,NaT,,Seibersdorf,,SP_01,inferred,Inferred from the logs: start is the first log...
1,d02,SP_02,2026-09-23 11:49:00,NaT,,Seibersdorf,,SP_01,inferred,Inferred from the logs: start is the first log...
2,d03,SP_03,2026-09-23 11:42:00,NaT,,Seibersdorf,,SP_01,inferred,Inferred from the logs: start is the first log...
3,d04,SP_04,2026-09-23 11:37:00,NaT,,Seibersdorf,,SP_01,inferred,Inferred from the logs: start is the first log...


In [ ]:
#| eval: false
evs

,time,node_id,event,source,notes
0,2026-09-24 10:15:00,,visit,inferred,All nodes resumed logging between 10:08 and 10...
1,2026-09-24 14:00:00,,visit,inferred,"SP_04, SP_01 and SP_03 resumed logging at 13:1..."
2,2026-09-28 12:00:00,,visit,inferred,All nodes resumed logging between 09:14 (SP_01...
3,2026-09-30 13:43:00,,visit,inferred,All nodes resumed logging between 13:41 and 13...
4,2026-09-30 13:52:00,SP_04,watering,inferred,Soil moisture rose from 15.0 to 25.5 between t...
5,2026-10-05 08:20:00,,visit,inferred,SP_01 and SP_03 resumed logging at 08:21 and 0...
6,2026-10-05 17:00:00,,card collection,inferred,"Collection of batch b01, whose logs end betwee..."
7,2026-10-07 10:45:00,SP_03,restart,inferred,Resumed logging after an outage from 04:40.
8,2026-10-08 10:46:00,SP_04,restart,inferred,Resumed logging after an outage from 23:30 on ...
9,2026-10-09 11:00:00,,card collection,inferred,"Collection of batch b02, whose logs end betwee..."


## Linking logs to deployments

`assign_deployments` adds the `deployment_id` and `plant_id` of each row. Rows that no deployment covers get missing values. Analyses can then group by deployment or plant instead of by `node_id`:

In [ ]:
#| export
def assign_deployments(
    df:pd.DataFrame, # Rows with `node_id` and `time`, such as the output of `measurements`
    deps:pd.DataFrame, # Output of `read_deployments`
)->pd.DataFrame: # `df` with `deployment_id` and `plant_id` columns
    "Attach each row to the deployment that covers its node and time"
    dep, plant = pd.Series(pd.NA, index=df.index, dtype='str'), pd.Series(pd.NA, index=df.index, dtype='str')
    for r in deps.itertuples():
        m = (df.node_id == r.node_id) & (df.time >= r.start)
        if pd.notna(r.end): m &= df.time < r.end
        dep[m], plant[m] = r.deployment_id, r.plant_id
    return df.assign(deployment_id=dep, plant_id=plant)

In [ ]:
deps_t = pd.DataFrame(dict(deployment_id=['d1', 'd2'], node_id=['A', 'A'], plant_id=['p1', 'p2'],
                           start=pd.to_datetime(['2026-10-01', '2026-10-03']), end=pd.to_datetime(['2026-10-03', None])))
df_t = pd.DataFrame(dict(node_id=['A', 'A', 'A', 'B'], time=pd.to_datetime(['2026-09-30', '2026-10-02', '2026-10-04', '2026-10-02'])))
test_eq(assign_deployments(df_t, deps_t).deployment_id.fillna('-').tolist(), ['-', 'd1', 'd2', '-'])

## Checking the logbook

`check_logbook` lists what needs a look. Each row is one issue:

- `unknown node`: the logs have a `node_id` that no deployment mentions.
- `overlap`: two deployments of one node overlap in time.
- `outside deployments`: logged rows that no deployment covers, with their count and time span.
- `scene change`: two neighbouring days of one deployment whose daily standard deviation maps correlate below `min_r`, from `scene_similarity`. A camera or plant probably moved. The logbook should then end the deployment there, or say why not.

The scene check uses only days with at least `min_n` measurements, and a low correlation can also come from a partial or unusual day. It flags days to review, not certain changes.

In [ ]:
#| export
def check_logbook(
    df:pd.DataFrame, # Output of `measurements`
    deps:pd.DataFrame, # Output of `read_deployments`
    min_r:float=0.5, # Correlation below which neighbouring days count as a scene change
    min_n:int=144, # Fewest measurements for a day to be compared
)->pd.DataFrame: # One row per issue
    "Issues between the logbook and the logs"
    rows = []
    for n in sorted(set(df.node_id) - set(deps.node_id)): rows.append(dict(check='unknown node', node_id=n, detail='No deployment for this node'))
    end = deps.end.fillna(pd.Timestamp.max)
    for n, g in deps.assign(end_=end).sort_values('start').groupby('node_id'):
        for a, b in zip(g.itertuples(), list(g.itertuples())[1:]):
            if b.start < a.end_: rows.append(dict(check='overlap', node_id=n, detail=f'{a.deployment_id} and {b.deployment_id} overlap'))
    out = assign_deployments(df, deps)
    for n, g in out[out.deployment_id.isna()].groupby('node_id'):
        rows.append(dict(check='outside deployments', node_id=n, detail=f'{len(g)} rows from {g.time.min():%Y-%m-%d %H:%M} to {g.time.max():%Y-%m-%d %H:%M}'))
    days = out.assign(day=out.time.dt.normalize()).groupby(['node_id', 'day']).deployment_id.agg(lambda s: s.iloc[0] if s.nunique() == 1 else None)
    for n, sim in scene_similarity(df, min_n).items():
        for a, b in zip(sim.index, sim.index[1:]):
            r, da, db = sim.loc[a, b], days.get((n, a)), days.get((n, b))
            if r < min_r and da is not None and da == db:
                rows.append(dict(check='scene change', node_id=n, detail=f'{a:%d %b} to {b:%d %b}: r = {r:.2f}, inside {da}'))
    return pd.DataFrame(rows, columns=['check', 'node_id', 'detail'])

In [ ]:
t = pd.date_range('2026-10-01', periods=4, freq='D')
df_t = pd.DataFrame(dict(node_id=['A', 'A', 'C', 'C'], time=t))
deps_t = pd.DataFrame(dict(deployment_id=['d1', 'd2'], node_id=['A', 'A'], plant_id=['p1', 'p1'],
                           start=pd.to_datetime(['2026-10-01 00:00', '2026-10-01 12:00']), end=pd.to_datetime([None, None])))
test_eq(set(check_logbook(pd.concat([df_t, pd.DataFrame(20., index=df_t.index, columns=px_cols)], axis=1), deps_t).check), {'unknown node', 'overlap', 'outside deployments'})

The checks on all logs collected so far, in `data/b02`:

In [ ]:
#| eval: false
mdf = measurements(clean(read_logs(sorted(Path('../data/b02').glob('dados_*.csv')))))
check_logbook(mdf, deps)

,check,node_id,detail
0,scene change,SP_01,"25 Sep to 28 Sep: r = 0.38, inside d01"
1,scene change,SP_01,"29 Sep to 30 Sep: r = 0.22, inside d01"
2,scene change,SP_01,"02 Oct to 05 Oct: r = 0.09, inside d01"
3,scene change,SP_02,"23 Sep to 24 Sep: r = -0.02, inside d02"
4,scene change,SP_02,"29 Sep to 30 Sep: r = 0.45, inside d02"
5,scene change,SP_02,"30 Sep to 01 Oct: r = 0.38, inside d02"
6,scene change,SP_03,"24 Sep to 26 Sep: r = -0.00, inside d03"
7,scene change,SP_03,"26 Sep to 29 Sep: r = 0.28, inside d03"
8,scene change,SP_03,"29 Sep to 30 Sep: r = 0.19, inside d03"
9,scene change,SP_04,"29 Sep to 30 Sep: r = 0.03, inside d04"
